# Comm-MAPOCA cloud training (Colab / Kaggle)
Trains a headless Linux build of a Comm-MAPOCA environment with the `comm_mapoca` trainer from https://github.com/Lihaj/4th-Year-Research-COMM-MAPOCA.

**Before using this notebook (once, on your local machine):**
1. Unity Hub -> your editor version -> Add modules -> **Linux Build Support (Mono)** + **Linux Dedicated Server Build Support**.
2. File -> Build Profiles -> platform **Linux Server** (Dedicated Server) -> add ONE scene (e.g. `PushBlockCollab` for the baseline, or `PushBlockCollabComm`) -> Build into a folder, e.g. `Builds/PushBlockCollab/`.
3. Zip the whole build folder and upload it: **Colab** -> your Google Drive; **Kaggle** -> create a private Dataset from the zip.

Tip: build BOTH the stock baseline scene and the comm scene as two zips; the same notebook runs either (change `ENV_ZIP`/`CONFIG`/`RUN_ID`).

In [ ]:
# 1) Hardware check
!nvidia-smi -L || echo 'no GPU (CPU session)'
!nproc && free -h | head -2

In [ ]:
# 2) Install the Comm-MAPOCA fork (contains trainer_type: comm_mapoca AND stock poca)
!git clone --depth 1 https://github.com/Lihaj/4th-Year-Research-COMM-MAPOCA.git repo
%pip install -q -e repo/ml-agents/ml-agents-envs -e repo/ml-agents/ml-agents
# verify the trainer registry
!python -c "from mlagents.plugins.trainer_type import get_default_trainer_types; t,_=get_default_trainer_types(); print(sorted(t))"

If the repo is **private**: `git clone https://<USERNAME>:<PERSONAL_ACCESS_TOKEN>@github.com/Lihaj/4th-Year-Research-COMM-MAPOCA.git repo` (create a fine-grained PAT on GitHub -> Settings -> Developer settings).

If pip complains about numpy/protobuf versions, restart the runtime once (Runtime -> Restart) and re-run from cell 3 - the installs persist within the session.

In [ ]:
# 3a) COLAB: mount Drive and unzip the environment build
from google.colab import drive
drive.mount('/content/drive')
ENV_ZIP = '/content/drive/MyDrive/CommMAPOCA/PushBlockCollab.zip'  # <- your zip
!mkdir -p env && unzip -q -o "$ENV_ZIP" -d env
!find env -name '*.x86_64' -exec chmod +x {} \;
!find env -name '*.x86_64'

In [ ]:
# 3b) KAGGLE alternative: the dataset is auto-mounted read-only under /kaggle/input
# !mkdir -p env && cp -r /kaggle/input/YOUR-DATASET-NAME/* env/
# !find env -name '*.x86_64' -exec chmod +x {} \; && find env -name '*.x86_64'

In [ ]:
# 4) Train (re-run this same cell after a disconnect: --resume continues from the last checkpoint)
ENV_BINARY = 'env/PushBlockCollab.x86_64'   # <- from the find output above
CONFIG     = 'repo/ml-agents/config/poca/PushBlockCollab.yaml'              # baseline
# CONFIG   = 'repo/ml-agents/config/comm_mapoca/PushBlockCollabComm.yaml'   # comm variant
RUN_ID     = 'pushblock_poca_baseline_cloud_s1'
NUM_ENVS   = 4   # parallel copies of the binary; lower to 2 if the CPU chokes

!mlagents-learn "$CONFIG" --env="$ENV_BINARY" --no-graphics \
  --num-envs=$NUM_ENVS --run-id="$RUN_ID" --torch-device=cuda --resume \
  || mlagents-learn "$CONFIG" --env="$ENV_BINARY" --no-graphics \
  --num-envs=$NUM_ENVS --run-id="$RUN_ID" --torch-device=cuda

In [ ]:
# 5) After (or during) training: archive results back to Drive (Colab)
!zip -q -r results_backup.zip results
!cp results_backup.zip /content/drive/MyDrive/CommMAPOCA/
# Kaggle: results/ persists in the notebook output automatically (Save Version).

## Session survival rules
- Checkpoints save every 500k steps; a disconnect loses at most that much. Just re-run cell 4 (`--resume` is already wired, with a first-run fallback).
- Back up `results/` to Drive (cell 5) before closing a session.
- To view curves locally: download the results zip and point TensorBoard at it.
- Multi-seed: open 2-3 sessions (or Kaggle+Colab in parallel), same cells, different `RUN_ID` (`..._s2`, `..._s3`).